# Kaggle Submission: Exp 016 Blend Exp015 Exp011

This notebook blends two already-produced submission CSV files directly.

Important:
- for Kaggle-downloaded scored submissions, `row_id` is not a reliable join key
- the safe assumption is that row order matters, so this notebook blends by row position
- `row_id` from the first file (`exp_015`) is passed through into the final CSV

Default first blend:
- `0.90 * exp_015 + 0.10 * exp_011`


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
WORK_ROOT.mkdir(parents=True, exist_ok=True)

EXP015_SUBMISSION_HINT = None  # Example: '/kaggle/input/my-dataset/submission.csv'
EXP011_SUBMISSION_HINT = None  # Example: '/kaggle/input/my-dataset/submission.csv'
W_EXP015 = 0.90
W_EXP011 = 0.10
BLEND_STRATEGY = 'weighted_mean'  # or 'rank_mean'
OUTPUT_PATH = WORK_ROOT / 'submission.csv'


def _csv_candidates_for_hint(hint: str | None) -> list[Path]:
    candidates: list[Path] = []

    if hint:
        direct = Path(hint)
        hint_name = direct.name.lower()
        if direct.is_file() and direct.suffix.lower() == '.csv':
            return [direct]
        if direct.is_dir():
            candidates.extend(sorted(direct.rglob('*.csv')))

        hinted = INPUT_ROOT / hint
        if hinted.is_file() and hinted.suffix.lower() == '.csv':
            return [hinted]
        if hinted.is_dir():
            candidates.extend(sorted(hinted.rglob('*.csv')))

    if INPUT_ROOT.exists():
        for path in INPUT_ROOT.rglob('*.csv'):
            if path.name == 'sample_submission.csv':
                continue
            if hint is None:
                if path.name == 'submission.csv':
                    candidates.append(path)
                continue
            path_l = str(path).lower()
            if hint.lower() in path_l or hint_name == path.name.lower():
                candidates.append(path)

    deduped = []
    seen = set()
    for path in candidates:
        key = str(path.resolve()) if path.exists() else str(path)
        if key in seen:
            continue
        seen.add(key)
        deduped.append(path)
    return deduped


def resolve_submission_file(label: str, hint: str | None) -> Path:
    candidates = _csv_candidates_for_hint(hint)
    if not candidates:
        raise FileNotFoundError(f'Could not find a matching CSV for {label}. Set the corresponding *_HINT.')
    if len(candidates) > 1 and hint is None:
        preview = [str(path) for path in candidates[:8]]
        raise RuntimeError(
            f'Multiple CSV candidates found for {label}. '
            f'Set the corresponding *_HINT to disambiguate. Candidates: {preview}'
        )
    return candidates[0]


def normalize_row_ids(values: pd.Series) -> pd.Series:
    out = values.astype(str).str.strip()
    out = out.str.replace('.ogg_', '_', regex=False)
    out = out.str.replace('.wav_', '_', regex=False)
    out = out.str.replace('.flac_', '_', regex=False)
    out = out.str.replace('.mp3_', '_', regex=False)
    return out


def normalize_weights(w_a: float, w_b: float) -> tuple[float, float]:
    total = float(w_a) + float(w_b)
    if total <= 0:
        raise ValueError('Blend weights must sum to a positive number.')
    return float(w_a) / total, float(w_b) / total


def load_submission(path: Path) -> tuple[pd.DataFrame, list[str]]:
    df = pd.read_csv(path)
    df = df.loc[:, ~df.columns.astype(str).str.startswith('Unnamed:')].copy()
    if 'row_id' not in df.columns:
        raise ValueError(f'{path} does not contain a row_id column.')

    species = [col for col in df.columns if col != 'row_id']
    if not species:
        raise ValueError(f'{path} does not contain prediction columns.')

    out = df[['row_id', *species]].reset_index(drop=True).copy()
    return out, species


def align_submissions(df_a: pd.DataFrame, df_b: pd.DataFrame, species_a: list[str], species_b: list[str]) -> tuple[pd.DataFrame, list[str], dict[str, int]]:
    if set(species_a) != set(species_b):
        only_a = sorted(set(species_a) - set(species_b))[:8]
        only_b = sorted(set(species_b) - set(species_a))[:8]
        raise ValueError(
            'Submission columns differ between the two files. '
            f'Only in A: {only_a}; only in B: {only_b}'
        )

    species = species_a.copy()
    df_b = df_b[['row_id', *species]].reset_index(drop=True).copy()

    if len(df_a) != len(df_b):
        raise ValueError(
            'Submission row counts differ between the two files. '
            f'Rows A: {len(df_a)}; Rows B: {len(df_b)}'
        )

    row_id_a = df_a['row_id'].astype(str).reset_index(drop=True)
    row_id_b = df_b['row_id'].astype(str).reset_index(drop=True)
    exact_matches = int((row_id_a == row_id_b).sum())
    norm_matches = int((normalize_row_ids(row_id_a) == normalize_row_ids(row_id_b)).sum())

    aligned = pd.DataFrame({'row_id': row_id_a})
    for col in species:
        aligned[f'{col}_a'] = df_a[col].to_numpy(dtype=np.float32, copy=False)
        aligned[f'{col}_b'] = df_b[col].to_numpy(dtype=np.float32, copy=False)

    diagnostics = {
        'rows': int(len(aligned)),
        'exact_row_id_matches': exact_matches,
        'normalized_row_id_matches': norm_matches,
    }
    return aligned, species, diagnostics


def rank_average(pred_a: np.ndarray, pred_b: np.ndarray) -> np.ndarray:
    rank_a = pd.DataFrame(pred_a).rank(axis=0, method='average', pct=True).to_numpy(dtype=np.float32)
    rank_b = pd.DataFrame(pred_b).rank(axis=0, method='average', pct=True).to_numpy(dtype=np.float32)
    return 0.5 * (rank_a + rank_b)


In [ ]:
exp015_path = resolve_submission_file('exp_015', EXP015_SUBMISSION_HINT)
exp011_path = resolve_submission_file('exp_011', EXP011_SUBMISSION_HINT)

sub_015_raw, species_015 = load_submission(exp015_path)
sub_011_raw, species_011 = load_submission(exp011_path)
aligned, species, diagnostics = align_submissions(sub_015_raw, sub_011_raw, species_015, species_011)
w015, w011 = normalize_weights(W_EXP015, W_EXP011)

print(json.dumps({
    'exp015_submission': str(exp015_path),
    'exp011_submission': str(exp011_path),
    'blend_strategy': BLEND_STRATEGY,
    'weight_exp015': w015,
    'weight_exp011': w011,
    **diagnostics,
    'species': int(len(species)),
}, indent=2))


In [ ]:
pred_015 = aligned[[f'{col}_a' for col in species]].to_numpy(dtype=np.float32, copy=False)
pred_011 = aligned[[f'{col}_b' for col in species]].to_numpy(dtype=np.float32, copy=False)

if BLEND_STRATEGY == 'weighted_mean':
    blended = (w015 * pred_015) + (w011 * pred_011)
elif BLEND_STRATEGY == 'rank_mean':
    rank_blend = rank_average(pred_015, pred_011)
    blended = (w015 * pred_015) + (w011 * pred_011)
    blended = 0.5 * blended + 0.5 * rank_blend
else:
    raise ValueError(f'Unsupported BLEND_STRATEGY: {BLEND_STRATEGY}')

submission = aligned[['row_id']].copy()
submission[species] = np.clip(blended, 0.0, 1.0)
submission.to_csv(OUTPUT_PATH, index=False)

corr = float(np.corrcoef(pred_015.ravel(), pred_011.ravel())[0, 1])
mean_abs_gap = float(np.mean(np.abs(pred_015 - pred_011)))
max_abs_gap = float(np.max(np.abs(pred_015 - pred_011)))

print(json.dumps({
    'submission_path': str(OUTPUT_PATH),
    'correlation_flat': corr,
    'mean_abs_gap': mean_abs_gap,
    'max_abs_gap': max_abs_gap,
}, indent=2))
submission.head()


## Notes

- This notebook blends by row position, not by `row_id`.
- That is intentional for Kaggle-downloaded scored submissions.
- First safe blend to try: `0.90 * exp_015 + 0.10 * exp_011`
- If it helps, next nearby checks are:
  - `0.95 / 0.05`
  - `0.85 / 0.15`
  - `rank_mean`
